In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

df = pd.read_csv("DataScience_QA.csv")

In [3]:
df

,Question,Answer
0,What is the purpose of feature engineering in ...,"Feature engineering involves selecting, transf..."
1,What are some common techniques for feature en...,Common techniques include one-hot encoding for...
2,What is the difference between batch processin...,Batch processing involves processing data in l...
3,What is natural language processing (NLP)?,Natural language processing is a field of arti...
4,What is sentiment analysis?,Sentiment analysis is the task of automaticall...
...,...,...
153,What is the role of rewards in reinforcement l...,Rewards in reinforcement learning serve as sig...
154,What is the policy gradient in reinforcement l...,The policy gradient is a method used to update...
155,What is the advantage of deep reinforcement le...,Deep reinforcement learning combines the benef...
156,What are some applications of reinforcement le...,Reinforcement learning is applied in various d...


In [4]:
df.shape

(158, 2)

In [6]:
df.isnull().sum()

Question    0
Answer      0
dtype: int64

# install needed models

In [26]:
! pip install langchain


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
! pip install  langchain-ollama


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [63]:
! pip install langchain-qdrant==0.1.4



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# import libraries

In [39]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings

In [52]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# document loader

In [53]:
loader = CSVLoader(file_path="DataScience_QA.csv")

data = loader.load()

print(data)

[Document(metadata={'source': 'DataScience_QA.csv', 'row': 0}, page_content='Question: What is the purpose of feature engineering in machine learning?\nAnswer: Feature engineering involves selecting, transforming, or creating new features from the raw data to improve the performance of machine learning models by making them more expressive, informative, and suitable for the task at hand.'), Document(metadata={'source': 'DataScience_QA.csv', 'row': 1}, page_content='Question: What are some common techniques for feature engineering?\nAnswer: Common techniques include one-hot encoding for categorical variables, scaling numerical features, creating interaction terms, binning or discretizing continuous variables, and extracting features from text or images using techniques like TF-IDF or CNNs.'), Document(metadata={'source': 'DataScience_QA.csv', 'row': 2}, page_content='Question: What is the difference between batch processing and real-time processing?\nAnswer: Batch processing involves pr

# split data

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)


texts = text_splitter.split_documents(data)

print(texts)

[Document(metadata={'source': 'DataScience_QA.csv', 'row': 0}, page_content='Question: What is the purpose of feature engineering in machine learning?'), Document(metadata={'source': 'DataScience_QA.csv', 'row': 0}, page_content='Answer: Feature engineering involves selecting, transforming, or creating new features from the raw'), Document(metadata={'source': 'DataScience_QA.csv', 'row': 0}, page_content='from the raw data to improve the performance of machine learning models by making them more'), Document(metadata={'source': 'DataScience_QA.csv', 'row': 0}, page_content='by making them more expressive, informative, and suitable for the task at hand.'), Document(metadata={'source': 'DataScience_QA.csv', 'row': 1}, page_content='Question: What are some common techniques for feature engineering?'), Document(metadata={'source': 'DataScience_QA.csv', 'row': 1}, page_content='Answer: Common techniques include one-hot encoding for categorical variables, scaling numerical'), Document(metadat

# Embeding models

In [55]:
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
)

In [56]:
embeddings

OllamaEmbeddings(model='qwen3-embedding:0.6b', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

# vector store 

In [ ]:
client = QdrantClient(":memory:")

vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists("test"):
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
vector_store = QdrantVectorStore(
    client=client,
    collection_name="test",
    embedding=embeddings,
)

In [ ]:
vector_store.add_documents(texts)

print("Documents stored successfully!")

## retriever

In [58]:
retriever = vector_store.as_retriever()

query = "What is machine learning?"
results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)

Question: What is unsupervised learning?
Question: What is the difference between supervised and unsupervised learning?
Answer: Reinforcement learning is a type of machine learning where an agent learns to make
Answer: Reinforcement learning is a type of machine learning where an agent learns to make


# LLM Model

In [ ]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2:1b")


I'll be happy to help you with your questions.

**Unsupervised Learning**

Unsupervised learning is a type of machine learning where the algorithm is not provided with any labels or target values. Instead, it's given an input dataset and must find patterns, relationships, or structure within that data on its own. The goal is to identify clusters, anomalies, or features that are relevant to the problem at hand.

**Supervised Learning**

Supervised learning, on the other hand, involves feeding the algorithm a labeled dataset where each sample has a corresponding output or target value. The task is to learn a mapping between input data and predicted outputs. This type of learning requires some prior knowledge about the relationship between variables, but it provides a clear objective to optimize.

**Overfitting in Machine Learning**

Overfitting occurs when a machine learning model is too complex and fits the training data too closely, resulting in poor performance on new, unseen data. In

# Questions

In [62]:
query = "Explain machine learning"
docs = retriever.invoke(query)

context = " ".join([doc.page_content for doc in docs])

response = llm.invoke(f"Answer based on context:\n{context}\n\nQuestion: {query}")

print(response)

I'd be happy to explain these concepts in detail.

**Unsupervised Learning**

Unsupervised learning is a type of machine learning where the algorithm is not given any labels or instructions about which data points belong together. The goal is to identify patterns, relationships, or clusters within the data without prior knowledge or supervision.

Think of it like a treasure hunt: you're not told which specific items are hidden somewhere in the data set, but you need to find all the hidden treasures (i.e., unique patterns) on your own. Unsupervised learning algorithms use various techniques such as clustering, dimensionality reduction, and association rules to identify these hidden patterns.

**Supervised Learning**

Supervised learning is a type of machine learning where the algorithm is given labeled data, which means it has been trained with examples that have a clear correct output for each input. The goal is to learn a mapping between inputs and outputs so that when new, unseen dat

# FULL CODE

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

#--------------DOCUMENT LOADER---------------

from langchain_community.document_loaders.csv_loader import CSVLoader

loader = CSVLoader(file_path="./example_data/mlb_teams_2012.csv") # put file path
data = loader.load()
print(data)

#----------------SPLIT DATAS------------------

! pip install -U langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0)
texts = text_splitter.split_text(document) # put input data(document --> data) and use split_text --> split document

#----------------EMBEDDING MODELS---------------

! pip install -qU langchain-ollama

from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="INSTALLED_MODEL",   
    # download embedding model in local using ollama or other platform and download in terminal or command prompt
)

#------------------VECTOR STORE-------------------

! pip install -qU langchain-qdrant

from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient


client = QdrantClient(":memory:")
vector_size = len(embeddings.embed_query("sample text"))

if not client.collection_exists("test"):
    client.create_collection(
        collection_name="test",
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
    
vector_store = QdrantVectorStore(
    client=client,
    collection_name="test",
    embedding=embeddings,
)   

#================================FROM LANGCHAIN DOCS======================

# ------------------ ADD DOCUMENTS (VERY IMPORTANT) ------------------

vector_store.add_documents(texts)

#-------------------RETREIVER-------------------------------

retriever = vector_store.as_retriever()

query = "What is machine learning?"
results = retriever.invoke(query)

for doc in results:
    print(doc.page_content)

